# Hey Jarvis — Intent Classification Model Training & Benchmark

This notebook builds the 8-class dataset (`PlayMusic`, `AddToPlaylist`, `PauseMusic`, `ResumeMusic`, `NextTrack`, `PreviousTrack`, `VolumeControl`, `OutOfScope`), trains both recommended models, and compares them side-by-side.

### Recommended Models
1. **TF-IDF (1–2 n-grams) + Logistic Regression (`models/logreg_intent.joblib`)**: Ultra-fast (< 1 ms) CPU baseline with strong linear separability on explicit playback commands.
2. **Fine-Tuned DistilBERT (`distilbert-base-uncased` -> `models/distilbert_intent/`)**: Best balance of semantic accuracy (distinguishing `PlayMusic` vs. `OutOfScope` like *"find the song..."* vs. *"play..."*) and low inference latency on GPU/CPU.
   - *Alternative Transformer options*: `microsoft/MiniLM-L12-H384-uncased` (smaller/faster on CPU) or `roberta-base` (slightly higher capacity if GPU memory allows).

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("Working dir:", os.getcwd())
print("CUDA available:", torch.cuda.is_available())

## 1. Generate Custom Playback Data & Build Merged Snips Dataset

In [ ]:
import runpy
import pandas as pd

runpy.run_path("nlp/generate_custom_data.py", run_name="__main__")
from nlp.dataset_builder import build_full_dataset
build_full_dataset()

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"data/processed/{split}.csv")
    print(f"{split}: {len(df)} rows | classes={df['label'].nunique()} | NaNs={df.isna().sum().to_dict()}")

## 2. Train Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
from nlp.train_baseline import train_baseline

baseline_model = train_baseline()

## 3. Fine-Tune Transformer Model (`distilbert-base-uncased`)

In [ ]:
from nlp.train_transformer import train_transformer

train_transformer(
    model_name="distilbert-base-uncased",
    max_length=32,
    learning_rate=3e-5,
    batch_size=32,
    num_epochs=4,
)

## 4. Evaluate & Compare Both Models Side-by-Side

In [ ]:
from nlp.evaluate_compare import evaluate_and_compare

report = evaluate_and_compare()
report

## 5. Quick Inference Sanity Check

In [ ]:
from nlp.predictor import IntentEngine, extract_music_query

engine = IntentEngine()
samples = [
    "play blinding lights by the weeknd",
    "find the song blinding lights",
    "pause the music",
    "skip to next track",
    "turn the volume down",
    "what is the weather in london",
]
for text in samples:
    res = engine.predict_both(text)
    print(f"\nInput: {text!r}")
    print("  LogReg    :", res["logreg"])
    print("  DistilBERT:", res["distilbert"])
    print("  Query     :", extract_music_query(text))